# Full-WAV spectrograms with separate time encodings

1. Run this notebook yourself to extract **all train and test WAVs** on this machine. Extraction uses CPU; no GPU, model downloads, ASR, scoring-model training or submission is needed.
2. Every full recording is preserved, including leading silence, pauses, fillers and background sound. Originals are read-only. Numerical spectrograms and sine/cosine coordinates are stored separately in one new HDF5 file; existing channel files are not edited.
3. Defaults: mono **16 kHz**, periodic Hann **25 ms**, hop **10 ms**, FFT **1024**, **128 HTK mel bands** from 20–8000 Hz, fixed-reference log power floor −100 dB. No per-record loudness normalization. Mono conversion averages channels; other rates are resampled.
4. Each recording stores `log_mel (T,128)`, `time_encoding (T,64)`, original frame start/center seconds `(T,)`, and valid sample fractions `(T,)`. A short final window uses documented right-zero-padding; recordings are never truncated or resized into images.
5. Time coordinates use the original frame-center seconds divided by 0.01, alternating sine/cosine with base 10000. Later training should project acoustic features and add or concatenate positional vectors; a patch model should encode patch-center timestamps. These arrays are not learned embeddings or a combined model input yet.
6. Labels and source CSV order are preserved; random test labels are ignored. SHA256 hashes bind records to original WAVs. Resuming validates saved feature checksums and settings before skipping completed WAVs. Changing settings requires a new output filename.
7. The output is several GB before compression. Progress is displayed per WAV. `complete=True` only after every expected recording is saved. Interrupt the extraction cell to stop; rerun to resume. A hard machine shutdown may damage an open HDF5 file; an interrupted cell is preferable.
8. At the end, inspect a saved spectrogram and its separate encoding. No training notebook is changed here.

In [ ]:
from pathlib import Path
import os
# CPU DSP only. Set before importing NumPy/SciPy to avoid excess BLAS threads.
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
CONFIG = {
    'DATA_ROOT': '/workspace/shl/data/audio_full',
    'LABEL_ROOT': None,  # None = train.csv and test.csv under DATA_ROOT.
    'OUTPUT': '/workspace/shl/audio_embeddings/shl_spectrogram_time_v1.h5',
    'LIMIT': None,  # None processes every train/test WAV. Integer = clearly partial smoke run.
    'MIN_FREE_GB': 2,
    'FEATURES': {
        'sample_rate': 16000, 'window_samples': 400, 'hop_samples': 160,
        'n_fft': 1024, 'n_mels': 128, 'f_min': 20.0, 'f_max': 8000.0,
        'db_floor': -100.0, 'pe_dimensions': 64, 'pe_base': 10000.0,
        'pe_reference_step_s': 0.01, 'compression': 'gzip', 'compression_level': 4,
    },
}
print('Output:', CONFIG['OUTPUT'])

## 1. Install required packages

1. This uses the current notebook kernel's Python. Existing compatible packages are reused. No PyTorch/CUDA installation is changed.
2. If you have already imported incompatible versions, restart the kernel after this cell, then Run All.

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "--disable-pip-version-check"] + ['numpy>=1.24,<3', 'scipy>=1.10,<2', 'pandas>=1.5,<4', 'soundfile>=0.12,<1', 'h5py>=3.8,<4', 'matplotlib>=3.6,<4', 'tqdm>=4.64,<5'])

## 2. Load the embedded extractor

1. This cell defines extraction functions. No WAV is processed until the next cell.
2. The notebook is self-contained; the accompanying Python script also supports command-line extraction.

In [ ]:
RUNTIME_SOURCE = '"""Full-WAV log-mel and separate sinusoidal time arrays; no model training."""\nimport argparse\nimport contextlib\nimport hashlib\nimport importlib.metadata\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport platform\nimport shutil\nimport tempfile\nfrom datetime import datetime, timezone\n\nimport h5py\nimport numpy as np\nimport pandas as pd\nfrom scipy import fft, signal\nimport soundfile as sf\nfrom tqdm.auto import tqdm\n\nVERSION = \'spectrogram-time-v1\'\nDEFAULTS = dict(sample_rate=16000, window_samples=400, hop_samples=160,\n                n_fft=1024, n_mels=128, f_min=20.0, f_max=8000.0,\n                db_floor=-100.0, pe_dimensions=64, pe_base=10000.0,\n                pe_reference_step_s=0.01, compression=\'gzip\', compression_level=4)\nARRAYS = [\'log_mel\', \'time_encoding\', \'frame_start_s\', \'frame_center_s\', \'valid_fraction\']\n\n\ndef sha(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(1024*1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef canonical(value):\n    return json.dumps(value, sort_keys=True, separators=(\',\', \':\'), allow_nan=False)\n\n\ndef validate_config(config):\n    c = dict(DEFAULTS, **config)\n    for key in [\'sample_rate\',\'window_samples\',\'hop_samples\',\'n_fft\',\'n_mels\',\'pe_dimensions\']:\n        if type(c[key]) is not int or c[key] <= 0:\n            raise ValueError(f\'{key} must be a positive integer\')\n    if not 0 < c[\'hop_samples\'] <= c[\'window_samples\'] <= c[\'n_fft\']:\n        raise ValueError(\'Require hop <= window <= FFT size\')\n    if not 0 <= c[\'f_min\'] < c[\'f_max\'] <= c[\'sample_rate\']/2:\n        raise ValueError(\'Invalid frequency range\')\n    if c[\'pe_dimensions\']%2 or c[\'pe_base\']<=1 or c[\'pe_reference_step_s\']<=0:\n        raise ValueError(\'Time encoding requires even dimensions, base>1 and positive time step\')\n    if c[\'compression\'] not in [\'gzip\',\'lzf\'] or not 0<=c[\'compression_level\']<=9:\n        raise ValueError(\'Unsupported compression settings\')\n    if not np.isfinite(c[\'db_floor\']) or c[\'db_floor\']>=0:\n        raise ValueError(\'db_floor must be finite and negative\')\n    return c\n\n\ndef filterbank(config):\n    c = config\n    to_mel = lambda hz: 2595*np.log10(1+np.asarray(hz)/700)\n    to_hz = lambda mel: 700*(10**(np.asarray(mel)/2595)-1)\n    edges = to_hz(np.linspace(to_mel(c[\'f_min\']),to_mel(c[\'f_max\']),c[\'n_mels\']+2))\n    frequencies = fft.rfftfreq(c[\'n_fft\'],1/c[\'sample_rate\'])\n    bank = np.maximum(0,np.minimum(\n        (frequencies[None,:]-edges[:-2,None])/(edges[1:-1]-edges[:-2])[:,None],\n        (edges[2:,None]-frequencies[None,:])/(edges[2:]-edges[1:-1])[:,None]))\n    if np.any(bank.sum(axis=1)==0):\n        raise ValueError(\'Empty mel filter: increase n_fft or reduce n_mels\')\n    return bank.astype(np.float32), edges[1:-1].astype(np.float32)\n\n\ndef time_encoding(times, config):\n    positions = np.asarray(times,dtype=np.float64)/config[\'pe_reference_step_s\']\n    angles = positions[:,None]/config[\'pe_base\']**(\n        np.arange(0,config[\'pe_dimensions\'],2,dtype=np.float64)/config[\'pe_dimensions\'])[None,:]\n    out = np.empty((len(times),config[\'pe_dimensions\']),dtype=np.float32)\n    out[:,0::2], out[:,1::2] = np.sin(angles), np.cos(angles)\n    return out\n\n\ndef extract_array(waveform, config):\n    c = validate_config(config)\n    x = np.asarray(waveform,dtype=np.float32)\n    if x.ndim!=1 or not len(x) or not np.isfinite(x).all():\n        raise ValueError(\'Require nonempty finite mono waveform\')\n    win,hop,sr = c[\'window_samples\'],c[\'hop_samples\'],c[\'sample_rate\']\n    count = max(1,1+math.ceil((len(x)-win)/hop))\n    padded_samples = (count-1)*hop+win-len(x)\n    padded = np.pad(x,(0,padded_samples))\n    frames = np.lib.stride_tricks.sliding_window_view(padded,win)[::hop]\n    window = signal.get_window(\'hann\',win,fftbins=True).astype(np.float32)\n    # Fixed reference, normalized FFT amplitude; no per-clip gain normalization.\n    spectrum = fft.rfft(frames*window,n=c[\'n_fft\'],axis=1)/window.sum()\n    power = (spectrum.real**2+spectrum.imag**2).astype(np.float32)\n    bank,_ = filterbank(c)\n    mel_power = power@bank.T\n    log_mel = (10*np.log10(np.maximum(mel_power,10**(c[\'db_floor\']/10)))).astype(np.float32)\n    starts = np.arange(count,dtype=np.float64)*hop/sr\n    centers = starts+win/(2*sr)\n    valid = np.minimum(win,np.maximum(0,len(x)-np.arange(count)*hop))/win\n    return dict(log_mel=log_mel,time_encoding=time_encoding(centers,c),\n                frame_start_s=starts,frame_center_s=centers,\n                valid_fraction=valid.astype(np.float32)), padded_samples\n\n\ndef discover(data_root, label_root=None):\n    root = Path(data_root).resolve()\n    label_root = Path(label_root).resolve() if label_root else root\n    rows, csv_hashes = [], {}\n    for split in [\'train\',\'test\']:\n        csv_path = label_root/f\'{split}.csv\'\n        table = pd.read_csv(csv_path,keep_default_na=False)\n        if \'filename\' not in table or table.filename.duplicated().any():\n            raise ValueError(f\'Invalid filename column in {csv_path}\')\n        wanted = set(table.filename)\n        if any(Path(name).name!=name or not name.lower().endswith(\'.wav\') for name in wanted):\n            raise ValueError(\'Expected plain WAV filenames in CSV\')\n        lookup = {}\n        for path in (root/split).rglob(\'*\'):\n            if path.suffix.lower()!=\'.wav\' or \'.ipynb_checkpoints\' in path.parts or path.name not in wanted:\n                continue\n            if path.name in lookup:\n                raise ValueError(f\'Duplicate source: {split}/{path.name}\')\n            lookup[path.name] = path\n        missing = sorted(wanted-set(lookup))\n        if missing:\n            raise FileNotFoundError(f\'{split}: {len(missing)} WAVs missing: {missing[:8]}\')\n        csv_hashes[f\'{split}.csv\'] = sha(csv_path)\n        for source_row, row in enumerate(table.to_dict(\'records\')):\n            filename = row[\'filename\']\n            label = float(row[\'label\']) if split==\'train\' else None\n            if label is not None and (not np.isfinite(label) or not 0<=label<=5):\n                raise ValueError(f\'Invalid original training label: {filename}\')\n            path = lookup[filename]\n            info = sf.info(path)\n            if info.frames<=0 or info.samplerate<=0:\n                raise ValueError(f\'Empty/invalid WAV: {path}\')\n            rows.append(dict(split=split,filename=filename,source_row=source_row,label=label,\n                             path=str(path),audio_sha256=sha(path),\n                             native_sample_rate=info.samplerate,native_channels=info.channels,\n                             native_samples=info.frames,duration_s=info.duration))\n    return rows,csv_hashes\n\n\ndef load_waveform(row, config):\n    if sha(row[\'path\']) != row[\'audio_sha256\']:\n        raise ValueError(\'WAV changed since preflight: \'+row[\'path\'])\n    x, rate = sf.read(row[\'path\'],dtype=\'float32\',always_2d=True)\n    if not np.isfinite(x).all():\n        raise ValueError(\'Nonfinite audio: \'+row[\'path\'])\n    x = x.mean(axis=1)\n    if rate != config[\'sample_rate\']:\n        gcd = math.gcd(rate,config[\'sample_rate\'])\n        x = signal.resample_poly(x,config[\'sample_rate\']//gcd,rate//gcd).astype(np.float32)\n    return x\n\n\ndef array_digest(values):\n    h = hashlib.sha256()\n    for key in ARRAYS:\n        a = np.ascontiguousarray(values[key])\n        h.update(key.encode());h.update(str(a.shape).encode());h.update(a.dtype.str.encode());h.update(a.tobytes())\n    return h.hexdigest()\n\n\ndef check_record(group, row, config):\n    if not group.attrs.get(\'complete\',False) or group.attrs[\'audio_sha256\']!=row[\'audio_sha256\']:\n        raise ValueError(\'Incomplete or mismatched saved record\')\n    arrays = {key:group[key][:] for key in ARRAYS}\n    n = len(arrays[\'frame_center_s\'])\n    if n==0 or arrays[\'log_mel\'].shape!=(n,config[\'n_mels\']) or arrays[\'time_encoding\'].shape!=(n,config[\'pe_dimensions\']):\n        raise ValueError(\'Invalid saved shapes\')\n    if any(not np.isfinite(a).all() for a in arrays.values()):\n        raise ValueError(\'Nonfinite saved features\')\n    if any(arrays[key].shape!=(n,) for key in [\'frame_start_s\',\'valid_fraction\']):\n        raise ValueError(\'Invalid frame metadata shape\')\n    if array_digest(arrays)!=group.attrs[\'array_sha256\']:\n        raise ValueError(\'Saved feature checksum mismatch\')\n\n\n@contextlib.contextmanager\ndef writer_lock(output):\n    # Keep lock inode on disk so another process cannot race through an unlinked lock.\n    import fcntl\n    with open(str(output)+\'.lock\',\'a\') as lock:\n        try:\n            fcntl.flock(lock,fcntl.LOCK_EX|fcntl.LOCK_NB)\n        except BlockingIOError as exc:\n            raise RuntimeError(\'Another extractor is writing this output\') from exc\n        try:\n            yield\n        finally:\n            fcntl.flock(lock,fcntl.LOCK_UN)\n\n\ndef source_code():\n    return Path(__file__).read_text()\n\n\ndef initialize(output, rows, csv_hashes, config, fingerprint):\n    handle,name = tempfile.mkstemp(prefix=output.name+\'.init-\',dir=output.parent)\n    os.close(handle)\n    temporary = Path(name)\n    try:\n        with h5py.File(temporary,\'w\') as f:\n            f.attrs.update(schema_version=1,version=VERSION,complete=False,\n                           fingerprint=fingerprint,config_json=canonical(config),\n                           created_at_utc=datetime.now(timezone.utc).isoformat(),\n                           expected_records=len(rows),completed_records=0,\n                           axis_order=\'time,mel_frequency\',time_encoding_axis_order=\'time,encoding_dimension\',\n                           normalization=\'No per-record peak/loudness normalization; later learned scaling must fit training folds only.\',\n                           log_mel_definition=\'10*log10(max(mel(abs(rfft(Hann*frame)/sum(Hann))**2),10**(db_floor/10))); HTK triangular filters, no area normalization, reference=1\',\n                           frame_policy=\'No centering or left padding; final partial window right-zero-padded; valid_fraction saved.\',\n                           pe_definition=\'position=original frame center seconds/pe_reference_step_s; alternating sin and cos; denominator=base**(2i/dimension). No PE is painted into acoustic values.\',\n                           test_label_policy=\'Test CSV labels are placeholders and ignored; stored NaN with label_known=False.\')\n            strings = h5py.string_dtype(\'utf-8\')\n            idx = f.create_group(\'index\')\n            for key in [\'filename\',\'split\',\'audio_sha256\']:\n                idx.create_dataset(key,data=[r[key] for r in rows],dtype=strings)\n            idx.create_dataset(\'source_row\',data=[r[\'source_row\'] for r in rows])\n            idx.create_dataset(\'label\',data=[np.nan if r[\'label\'] is None else r[\'label\'] for r in rows])\n            idx.create_dataset(\'label_known\',data=[r[\'label\'] is not None for r in rows])\n            idx.create_dataset(\'completed\',data=np.zeros(len(rows),dtype=bool))\n            _,centers = filterbank(config)\n            f.create_dataset(\'mel_centers_hz\',data=centers)\n            f.create_dataset(\'mel_filterbank\',data=filterbank(config)[0])\n            f.create_dataset(\'source_manifest_json\',data=canonical(dict(rows=rows,csv_sha256=csv_hashes)),dtype=strings)\n            f.create_dataset(\'extractor_source\',data=source_code(),dtype=strings)\n            versions = {package:importlib.metadata.version(package) for package in [\'numpy\',\'scipy\',\'pandas\',\'soundfile\',\'h5py\',\'tqdm\']}\n            f.create_dataset(\'environment_json\',data=canonical(dict(python=platform.python_version(),packages=versions)),dtype=strings)\n            f.create_group(\'records\');f.create_group(\'pending\');f.flush()\n        temporary.replace(output)\n    finally:\n        temporary.unlink(missing_ok=True)\n\n\ndef run(data_root, output, label_root=None, config=None, limit=None, min_free_gb=2):\n    c = validate_config(config or {})\n    output = Path(output).resolve();output.parent.mkdir(parents=True,exist_ok=True)\n    if limit is not None and (type(limit) is not int or limit<=0):\n        raise ValueError(\'limit must be None or a positive integer\')\n    print(\'Preflight: matching source CSVs, reading WAV headers and hashing all sources...\',flush=True)\n    rows,csv_hashes = discover(data_root,label_root)\n    identity = [{k:r[k] for k in [\'split\',\'filename\',\'source_row\',\'label\',\'audio_sha256\']} for r in rows]\n    fingerprint = hashlib.sha256(canonical(dict(version=VERSION,config=c,rows=identity,source_sha256=hashlib.sha256(source_code().encode()).hexdigest())).encode()).hexdigest()\n    print(f\'Found {len(rows)} WAVs; train={sum(r["split"]=="train" for r in rows)}, test={sum(r["split"]=="test" for r in rows)}. Output: {output}\',flush=True)\n    with writer_lock(output):\n        if not output.exists():\n            initialize(output,rows,csv_hashes,c,fingerprint)\n        with h5py.File(output,\'r+\') as f:\n            if f.attrs.get(\'fingerprint\')!=fingerprint:\n                raise ValueError(\'Existing output belongs to different inputs/settings/code. Select a new OUTPUT filename; original file was preserved.\')\n            for key in list(f[\'pending\']):\n                del f[\'pending\'][key]\n            completed = np.zeros(len(rows),dtype=bool)\n            for i,row in enumerate(rows):\n                key = f\'{i:06d}\'\n                if key in f[\'records\']:\n                    check_record(f[\'records\'][key],row,c)\n                    completed[i] = True\n            f[\'index/completed\'][:] = completed\n            f.attrs[\'completed_records\'] = int(completed.sum())\n            f.attrs[\'complete\'] = bool(completed.all());f.flush()\n            targets = list(range(len(rows)))[:limit] if limit else list(range(len(rows)))\n            for i in tqdm(targets,desc=\'Full-WAV spectrograms\',unit=\'wav\'):\n                if completed[i]:\n                    continue\n                row = rows[i]\n                # Upper bound for one uncompressed record plus user\'s reserve.\n                samples = math.ceil(row[\'duration_s\']*c[\'sample_rate\'])\n                frames = max(1,1+math.ceil((samples-c[\'window_samples\'])/c[\'hop_samples\']))\n                budget = frames*(4*(c[\'n_mels\']+c[\'pe_dimensions\']+1)+16)\n                if shutil.disk_usage(output.parent).free < min_free_gb*1024**3+budget:\n                    raise OSError(\'Disk reserve reached; output retains completed recordings. Free space and rerun.\')\n                x = load_waveform(row,c);values,padding = extract_array(x,c)\n                key = f\'{i:06d}\';g = f[\'pending\'].create_group(key)\n                g.attrs.update(filename=row[\'filename\'],split=row[\'split\'],audio_sha256=row[\'audio_sha256\'],\n                               source_row=row[\'source_row\'],native_sample_rate=row[\'native_sample_rate\'],\n                               native_channels=row[\'native_channels\'],resampled_samples=len(x),\n                               duration_s=len(x)/c[\'sample_rate\'],right_padding_samples=padding,\n                               array_sha256=array_digest(values),complete=False)\n                options = dict(compression=c[\'compression\'],shuffle=True)\n                if c[\'compression\']==\'gzip\':\n                    options[\'compression_opts\'] = c[\'compression_level\']\n                for name,array in values.items():\n                    g.create_dataset(name,data=array,**options)\n                g.attrs[\'complete\'] = True;f.flush()\n                f.move(f\'pending/{key}\',f\'records/{key}\');f.flush()\n                completed[i] = True;f[\'index/completed\'][i] = True\n                f.attrs[\'completed_records\'] = int(completed.sum());f.flush()\n            f.attrs[\'complete\'] = bool(completed.all())\n            f.attrs[\'last_run_utc\'] = datetime.now(timezone.utc).isoformat();f.flush()\n            print(f\'Saved {int(completed.sum())}/{len(rows)} recordings; complete={bool(completed.all())}; size={output.stat().st_size/1024**3:.3f} GiB\',flush=True)\n    return output\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--data-root\',default=\'/workspace/shl/data/audio_full\')\n    parser.add_argument(\'--label-root\')\n    parser.add_argument(\'--output\',default=\'/workspace/shl/audio_embeddings/shl_spectrogram_time_v1.h5\')\n    parser.add_argument(\'--limit\',type=int,default=None)\n    args = parser.parse_args()\n    run(args.data_root,args.output,args.label_root,limit=args.limit)\n\n\nif __name__ == \'__main__\':\n    main()\n'
import importlib.util, tempfile
RUNTIME_PATH = Path(tempfile.mkdtemp(prefix='shl-spectrogram-runtime-'))/'extract_spectrograms.py'
RUNTIME_PATH.write_text(RUNTIME_SOURCE)
spec = importlib.util.spec_from_file_location('shl_spectrogram_runtime', RUNTIME_PATH)
runtime = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runtime)
print('Extractor ready:', runtime.VERSION)

## 3. Extract every WAV and save on this machine

1. The preflight requires every source WAV listed in both CSVs, excluding Jupyter checkpoints and unrelated files. It rejects duplicates and missing files before writing results.
2. Rerun this cell to resume a compatible output. It validates existing records and prints completion status.

In [ ]:
OUTPUT = runtime.run(CONFIG['DATA_ROOT'], CONFIG['OUTPUT'],
    label_root=CONFIG['LABEL_ROOT'], config=CONFIG['FEATURES'],
    limit=CONFIG['LIMIT'], min_free_gb=CONFIG['MIN_FREE_GB'])

## 4. Inspect the saved representation

1. Change `PREVIEW_FILENAME` and `PREVIEW_SPLIT` to inspect another saved recording.
2. The two plots are separate: acoustic energy versus time, and sine/cosine coordinates versus time. Neither is painted onto the other.

In [ ]:
import h5py, numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, FileLink
PREVIEW_FILENAME = 'audio_577.wav'
PREVIEW_SPLIT = 'train'
with h5py.File(OUTPUT, 'r') as f:
    print('Complete:', bool(f.attrs['complete']), '| Saved:', f.attrs['completed_records'], '/', f.attrs['expected_records'])
    names = f['index/filename'].asstr()[:]
    splits = f['index/split'].asstr()[:]
    matches = np.flatnonzero((names == PREVIEW_FILENAME) & (splits == PREVIEW_SPLIT))
    if not len(matches):
        raise ValueError('Preview filename/split not in source index')
    key = f'{int(matches[0]):06d}'
    if key not in f['records']:
        print('Selected preview is not extracted yet; choose a completed recording or resume extraction.')
    else:
        g = f['records'][key]
        mel, pe, times = g['log_mel'][:], g['time_encoding'][:], g['frame_center_s'][:]
        print('Log-mel:', mel.shape, '| Encoding:', pe.shape, '| Duration:', g.attrs['duration_s'])
        fig, axes = plt.subplots(2, 1, figsize=(15, 7), sharex=True)
        extent = [times[0]-0.005, times[-1]+0.005, 0, mel.shape[1]]
        im = axes[0].imshow(mel.T, origin='lower', aspect='auto', extent=extent, vmin=-100, vmax=0)
        fig.colorbar(im, ax=axes[0], label='Fixed-reference log power (dB)')
        axes[0].set(ylabel='Mel band index', title=f'{PREVIEW_SPLIT}/{PREVIEW_FILENAME}: full recording')
        axes[1].imshow(pe.T, origin='lower', aspect='auto', extent=[extent[0],extent[1],0,pe.shape[1]], vmin=-1, vmax=1, cmap='coolwarm')
        axes[1].set(xlabel='Original recording time (seconds)', ylabel='Encoding dimension', title='Separate sinusoidal time coordinates')
        fig.tight_layout(); plt.show()
display(FileLink(str(OUTPUT)))